# The Idea
Supply responsivess changes with reform. We have no data to model how this will happen, we can only rely on the literature and **exogenously** impose a value. Key sources are:

+ Hilber and Vermeulen (2016)
+ Drayton, Levell and Sturrock (2024)
+ Ball, Meen and Neygaard (2010)

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from python.functions.bridge import run_bridge, run_scenario, backtest_one_step, build_bridge_inputs

In [ ]:
# Loading ARDL params
coefs = pd.read_csv("../../R/models/ardl_coefs_full.csv")
ecm = coefs[coefs["type"] == "ecm"].set_index("term")["estimate"]
lr = coefs[coefs["type"] == "longrun"].set_index("term")["estimate"]
    
# england master for init
master = pd.read_csv("../../data/python_master/england_master.csv")
master.columns.values[0] = "period"

master["lhstarts"] = np.log(master["starts"])
master["lrprc"]    = np.log(master["hprice"] / master["gdp_def"])
master["lrcc"]     = np.log(master["cc"] / master["gdp_def"])
master["lvol"]     = np.log(master["vol"])
master["r3"]       = master["rate"]

init = master[master["period"].isin(["2025Q1", "2025Q2", "2025Q3", "2025Q4"])][
    ["period", "lhstarts", "lrprc", "lvol", "r3", "lrcc"]
].reset_index(drop=True)

obr = pd.read_csv("../../data/python_master/OBR/obr_scenario.csv")
obr = obr[obr["period"] != "2025Q4"].reset_index(drop=True)  # drop overlap anchor row


# Diagnostic to see if ECM is constructed correctly
for q in ["2023Q1", "2023Q2", "2023Q3", "2023Q4",
          "2024Q1", "2024Q2", "2024Q3", "2024Q4",
          "2025Q1", "2025Q2", "2025Q3"]:
    backtest_one_step(master, q, ecm, lr)
    print()

In [ ]:
SCENARIOS = {
    "baseline":         {},
    "lrprc_uplift_25":  {"lrprc": 1.25},
    "lrprc_uplift_50":  {"lrprc": 1.50},
    "lrprc_downlift_25": {"lrprc": 0.75},
    "lrprc_downlift_50": {"lrprc": 0.50},

    "lvol_uplift_25":   {"lvol": 1.25},
    "lvol_uplift_50":   {"lvol": 1.50},
    "lvol_downlift_25": {"lvol": 0.75},
    "lvol_downlift_50": {"lvol": 0.50},

    "joint_uplift_50": {"lrprc": 1.5, "lvol": 1.5}, 
}

starts = {name: run_scenario(mult, init, obr, ecm, lr) for name, mult in SCENARIOS.items()}

comparison = pd.DataFrame({name: df.set_index("period")["lhstarts"] for name, df in starts.items()})
display(comparison.head())

B = build_bridge_inputs()
OUT_DIR = "../../data/outputs"
CF_DIR = f"{OUT_DIR}/elasticity_cf"

for name, df in starts.items():
    out = df[["period", "lhstarts"]].copy()
    out["period"] = out["period"].astype(str)
    out.to_csv(f"{CF_DIR}/{name}.csv", index=False)

cf = {name: run_bridge(df, "lhstarts", B["model"], B["smearing"], B["hist_ln_C"], B["hist_ln_S"],
                       B["fy_map"], B["net_add"], B["actual_back"])
      for name, df in starts.items()}

target = 1_500_000
for name, d in cf.items():
    cumulative = sum(d.values())
    print(f"{name}: {cumulative:,.0f} ({100*cumulative/target:.1f}% of target, "
          f"shortfall {target-cumulative:,.0f})")

cumulatives = {name: sum(d.values()) for name, d in cf.items()}

for var in ["lrprc", "lvol"]:
    var_names = {n for n in SCENARIOS if n.startswith(var)} | {"baseline"}
    var_cumulatives = {k: v for k, v in cumulatives.items() if k in var_names}
    spread = max(var_cumulatives.values()) - min(var_cumulatives.values())
    print(f"\n{var} ±25/50% range: {spread:,.0f} homes "
          f"({max(var_cumulatives, key=var_cumulatives.get)} to "
          f"{min(var_cumulatives, key=var_cumulatives.get)})")

In [ ]:
import matplotlib.pyplot as plt

anchor = init.iloc[-1]["lrprc"]  # 2025Q4 real house price level (log, gdp_def-deflated)

# Restrict to the actual forecast horizon used in the bridge (2026Q1-2029Q1)
horizon = obr[obr["period"] <= "2029Q1"].copy()
horizon["gap_pct"] = 100 * (horizon["lrprc"] - anchor)

print(f"2025Q4 anchor (log lrprc): {anchor:.4f}")
print(f"Mean gap over horizon:     {horizon['gap_pct'].mean():+.2f}%")
print(f"Min/max gap:               {horizon['gap_pct'].min():+.2f}% / {horizon['gap_pct'].max():+.2f}%")

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(horizon["period"], horizon["gap_pct"], color="tab:green")
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(horizon["gap_pct"].mean(), color="grey", linestyle="--", linewidth=1,
           label=f"Mean: {horizon['gap_pct'].mean():+.1f}%")

ax.set_ylabel("Real house price vs 2025Q4 (%)")
ax.set_xlabel("Period")
ax.set_title("OBR real house price forecast relative to 2025Q4 anchor")
ax.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Result
Raising long-run supply price elasticity leaves cumulative net additions essentially unchanged, difference only in the thousands. The reason is that the OBR demand path holds real house prices close to flat over the forecast horizon so the elasticity channel has no way to work. A more elastic supply curve does not do much if there is no movement along the curve.


The results agrees with the reform scenario: whether we have more units (OBR's ~170k) or higher supply price responsiveness the 1.5m target is missed.